# ECG Waveform Analysis

## Structured Extraction from Paper ECGs — JSL-30B vs Claude Sonnet 4.6 vs GPT-5.4

**Task**: Extract cardiac measurements (HR, PR, QRS, QT, axis) from scanned 12-lead ECG images and evaluate against machine-computed ground truth.

**Dataset**: [`PULSE-ECG/ECGBench`](https://huggingface.co/datasets/PULSE-ECG/ECGBench) — 60 ECG images from PTB-XL
**Ground Truth**: [PTB-XL+](https://physionet.org/content/ptb-xl-plus/) 12SL algorithm measurements (17,991 records)
**Models**: JSL Medical VL 30B (on-prem H100) · Claude Sonnet 4.6 · GPT-5.4

In [ ]:
# -- Colab Setup ---------------------------------------------------------------
# Run this cell once to install dependencies and download data.
# Skipped automatically when running locally.
import os, sys
if "COLAB_RELEASE_TAG" in os.environ:
    # 1) Python dependencies
    !pip install -q pandas numpy tqdm pillow datasets matplotlib seaborn scikit-learn openai python-dotenv faiss-cpu requests PyMuPDF

    # 2) Source files (utils/, config)
    !wget -qO git_downloads.tar.gz "https://ckl-emr-bucket.s3.us-east-1.amazonaws.com/vlm-workshop/git_downloads.tar.gz"
    !tar xzf git_downloads.tar.gz && rm git_downloads.tar.gz

    # 3) Cached predictions + datasets
    !wget -qO nb7_data.tar.gz "https://ckl-emr-bucket.s3.us-east-1.amazonaws.com/vlm-workshop/nb7_data.tar.gz"
    !tar xzf nb7_data.tar.gz && rm nb7_data.tar.gz

    print("Setup complete — all dependencies and data downloaded.")


In [ ]:
import os, json, time, base64, warnings
from pathlib import Path
from collections import Counter, defaultdict
from io import BytesIO
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
from IPython.display import HTML, display
import requests

from utils.config import OUTPUT_DIR
from utils import *

# ── Configuration ────────────────────────────────────────────────────────────
NB_NAME    = "nb7_ecg_waveform_analysis"
USE_CACHE  = True

# ── Model: JSL-30B-VL-V3 on jsl-gpu H100 ────────────────────────────────────
VLM_BASE_URL = "http://localhost:9465/v1"
VLM_MODEL    = "jsl-medical-vl-30b"
VLM_API_KEY  = None
set_cache_model(VLM_MODEL)

# ── Paths ────────────────────────────────────────────────────────────────────
ECG_N = 60
COMPARE_MODELS = ["jsl-medical-vl-30b", "anthropic/claude-sonnet-4.6", "openai/gpt-5.4"]

print(f"Model   : {VLM_MODEL}")
print(f"Endpoint: {VLM_BASE_URL}")
print(f"Cache   : {'ON' if USE_CACHE else 'OFF'} (model={VLM_MODEL})")

In [ ]:
if not USE_CACHE:
    check_server(VLM_BASE_URL, VLM_MODEL, api_key=VLM_API_KEY)

---
## Section 1: 12-Lead ECG Dataset

Paper ECGs contain 12 leads recorded simultaneously — 4 limb leads (I, II, III, aVR, aVL, aVF) and 6 precordial leads (V1-V6). Each waveform encodes cardiac measurements readable from the grid paper.

In [ ]:
bench_images, bench_ids, bench_ecg_ids, bench_classes, gt_data = load_ecg_docs(ECG_N)
preview_ecg_docs(bench_images, bench_ids, bench_classes, gt_data)

---
## Section 2: Extraction + Triage Schema

A single schema extracts **5 measurable cardiac parameters** and a **normal/abnormal triage** in one inference call.

| Field | What it measures | Normal range | Unit |
|-------|-----------------|--------------|------|
| `heart_rate_bpm` | Ventricular rate from R-R intervals | 60-100 | bpm |
| `pr_interval_ms` | Atrial-to-ventricular conduction time | 120-200 | ms |
| `qrs_duration_ms` | Ventricular depolarization time | 80-120 | ms |
| `qt_interval_ms` | Total ventricular electrical activity | 350-450 | ms |
| `qrs_axis_degrees` | Mean QRS vector direction | -30 to +90 | degrees |
| `screening_result` | Normal vs abnormal triage | — | — |
| `confidence` | Model confidence in triage | — | — |

In [ ]:
ECG_SCHEMA = {
    "type": "object",
    "properties": {
        "heart_rate_bpm":    {"type": "integer"},
        "pr_interval_ms":    {"type": "integer"},
        "qrs_duration_ms":   {"type": "integer"},
        "qt_interval_ms":    {"type": "integer"},
        "qrs_axis_degrees":  {"type": "integer"},
        "rhythm":            {"type": "string"},
        "screening_result":  {"type": "string"},
        "confidence":        {"type": "string"},
        "findings":          {"type": "array", "items": {"type": "string"}},
        "clinical_summary":  {"type": "string"},
    },
    "required": ["heart_rate_bpm", "pr_interval_ms", "qrs_duration_ms", "qt_interval_ms",
                  "qrs_axis_degrees", "rhythm", "screening_result", "confidence",
                  "findings", "clinical_summary"],
    "additionalProperties": False,
}

ECG_PROMPT = """You are a cardiologist reading a 12-lead ECG paper tracing.
Extract cardiac measurements and provide a clinical assessment.

Measurements (read from the grid paper):
- heart_rate_bpm: count R-R intervals, each small box = 0.04s at 25mm/s
- pr_interval_ms: from start of P wave to start of QRS complex
- qrs_duration_ms: width of QRS complex
- qt_interval_ms: from start of QRS to end of T wave
- qrs_axis_degrees: estimate from limb leads (frontal plane, -180 to +180)
- rhythm: describe the underlying rhythm

Clinical assessment:
- screening_result: "normal" if regular sinus rhythm with no abnormalities, "abnormal" if any pathology
- confidence: high, medium, or low
- findings: list all specific observations
- clinical_summary: one-sentence overall interpretation

Return ACTUAL measurements from this specific ECG, not templates."""

print(f"Schema: {len(ECG_SCHEMA['properties'])} fields")
print(f"Required: {ECG_SCHEMA['required']}")

---
## Section 3: Structured Extraction

Run the VLM on all ECGBench images — one call per image extracts both cardiac measurements and normal/abnormal triage.

In [ ]:
extraction_results = run_ecg_extraction_cached(
    NB_NAME, "merged_results", USE_CACHE, bench_images, bench_ids, bench_ecg_ids,
    bench_classes, gt_data, ECG_PROMPT, ECG_SCHEMA,
    VLM_BASE_URL, VLM_MODEL, VLM_API_KEY)

In [ ]:
display_ecg_extraction(bench_images, extraction_results, gt_data)

---
## Section 4: Extraction Accuracy — JSL-30B vs Claude Sonnet 4.6 vs GPT-5.4

**Mean Absolute Error (MAE)** measures how close VLM-extracted values are to machine-computed ground truth.
Lower is better — ±0 means perfect extraction.

| Metric | What it means |
|--------|--------------|
| MAE | Average error across all samples |
| Median AE | Typical error (robust to outliers) |
| Within 10/15/20% | % of predictions within that tolerance of ground truth |

In [ ]:
show_ecg_mae(extraction_results, VLM_MODEL)

In [ ]:
all_results = load_ecg_model_results(NB_NAME, "merged_results", COMPARE_MODELS)
set_cache_model(VLM_MODEL)
show_ecg_mae_comparison(all_results)

In [ ]:
plot_ecg_scatter(all_results)

In [ ]:
show_ecg_comparison(extraction_results)

---
## Section 5: Binary Triage — JSL-30B vs Claude Sonnet 4.6 vs GPT-5.4

The same extraction call also yields a **normal/abnormal screening** — no extra inference needed. We compare triage accuracy (normal vs abnormal) across all three models against PTB-XL diagnostic labels.

In [ ]:
display_ecg_triage_from_merged(bench_images, extraction_results)

In [ ]:
show_merged_triage_accuracy(extraction_results)

In [ ]:
show_triage_comparison(all_results)

---
## ECG Digitization: Impact & ROI

### What This Delivers

| Metric | Manual Transcription | JSL-30B (on-prem) | Claude Sonnet 4.6 | GPT-5.4 |
| --- | --- | --- | --- | --- |
| Time per ECG | 5-10 min | < 2 sec | ~9 sec | ~4 sec |
| Batch of 60 ECGs | 5-10 hours | < 2 min | ~9 min | ~4 min |
| Cost per ECG | $5-15 (technician) | ~$0 (self-hosted) | ~$0.03 (API) | ~$0.02 (API) |
| Measurements | Depends on reader | HR, PR, QRS, QT, axis | HR, PR, QRS, QT, axis | HR, PR, QRS, QT, axis |

### Use Cases

- **Hospital digitization**: Convert paper ECG archives into structured, searchable data
- **Telemedicine**: Field clinics photograph ECGs, get instant measurements
- **Clinical research**: Extract features from thousands of historical ECGs for retrospective studies
- **On-prem advantage**: JSL-30B runs on your own infrastructure — no data leaves the building

In [ ]:
ecg_out = OUTPUT_DIR / "ecg"
ecg_out.mkdir(parents=True, exist_ok=True)

path = ecg_out / "ecg_results.json"
with open(path, "w") as f:
    json.dump(extraction_results, f, indent=2, default=str)
print(f"ecg_results.json: {path.stat().st_size/1024:.1f} KB")